## Load Data Set

In [1]:
import pandas as pd
import matplotlib.pyplot as plt

deliveries = pd.read_csv('../data/raw/deliveries.csv')
routes = pd.read_csv('../data/raw/routes.csv')

## Cleaning Data and Mearging

In [2]:
deliveries.duplicated().sum()

np.int64(1)

In [3]:
deliveries.drop_duplicates(inplace=True)

In [ ]:
merged = deliveries.merge(routes, on='route_id', how='left')

print(merged.shape)   
merged.head()

(12, 8)


,record_id,month,route_id,hub,promised_days,actual_days,route,service_type
0,1,Jan,R1,Mumbai,2,2,Metro Link,Express
1,2,Jan,R2,Chennai,3,4,City Dash,Express
2,3,Jan,R3,Delhi,5,8,Highway Freight,Standard
3,4,Jan,R4,Mumbai,6,10,Rural Feeder,Standard
4,5,Feb,R1,Chennai,2,5,Metro Link,Express


In [7]:
merged.isnull().sum()

record_id        0
month            0
route_id         0
hub              0
promised_days    0
actual_days      0
route            0
service_type     0
dtype: int64

## New Column and Groupby Summery

In [8]:
merged['delay_days'] = merged['actual_days'] - merged['promised_days']

In [9]:
merged['is_delayed'] = merged['actual_days'] > merged['promised_days']

summary = (
    merged.groupby('service_type')
    .agg(
        total_delay_days=('delay_days', 'sum'),
        delay_incidence_rate=('is_delayed', lambda s: s.mean() * 100)
    )
    .round(2)
    .reset_index()
)
summary

,service_type,total_delay_days,delay_incidence_rate
0,Express,12,66.67
1,Standard,22,83.33


In [10]:
route_delay = merged.groupby(['route_id', 'route'])['delay_days'].sum()

top_route = route_delay.idxmax()          # (route_id, route)
top_route_delay = route_delay.max()
total_delay = merged['delay_days'].sum()
share_pct = round(top_route_delay / total_delay * 100, 2)

print(f"Top route: {top_route[0]} ({top_route[1]})")
print(f"Route delay_days: {top_route_delay}")
print(f"Overall delay_days: {total_delay}")
print(f"Share of total: {share_pct}%")

Top route: R4 (Rural Feeder)
Route delay_days: 14
Overall delay_days: 34
Share of total: 41.18%


## Chart and Exporting clean Data